# 🏷️ Fase 4 — Quem, o Quê e Onde?
## Reconhecimento de Entidades Nomeadas (NER) em Português

---
**Roteiro-Desafio-NLP · Fatec DSM 6º Semestre · 2026**  
**Grupo:** ___________________________  **Data:** ___/___/______

---

### 🎯 Objetivo desta fase
Aplicar o NorBERTo para **NER (Named Entity Recognition)** — identificar e classificar entidades como pessoas, organizações, datas e valores monetários em textos jornalísticos e regulatórios em português.

### 📚 O que você vai aprender
- Diferença entre classificação de sequência e classificação de tokens
- O que é o formato BIO/IOB de anotação de entidades
- Como usar um modelo de NER pré-treinado
- Como construir uma interface de visualização de entidades

---


## 📦 Passo 1 — Instalação

In [ ]:
!pip install transformers datasets torch spacy -q
!python -m spacy download pt_core_news_lg -q
print("✅ Pronto!")


## 📖 Passo 2 — O que é NER e o formato BIO?
**NER** é a tarefa de identificar entidades em texto e classificá-las em categorias.

O formato **BIO** (Beginning-Inside-Outside) anota cada token:
- **B-TIPO**: início de uma entidade do TIPO
- **I-TIPO**: continuação de uma entidade do TIPO  
- **O**: token que não é entidade

Exemplo: *"O Banco Central do Brasil aumentou os juros"*
- O → O
- Banco → B-ORG
- Central → I-ORG
- do → I-ORG
- Brasil → I-ORG
- aumentou → O
- os → O
- juros → O


In [ ]:
# Demonstração visual do formato BIO
texto_exemplo = "O Banco Central do Brasil anunciou em maio de 2026 que a taxa Selic subiu para 10,5%."
tokens_bio = [
    ("O", "O"), ("Banco", "B-ORG"), ("Central", "I-ORG"), ("do", "I-ORG"),
    ("Brasil", "I-ORG"), ("anunciou", "O"), ("em", "O"),
    ("maio", "B-DATE"), ("de", "I-DATE"), ("2026", "I-DATE"),
    ("que", "O"), ("a", "O"), ("taxa", "O"), ("Selic", "B-MISC"),
    ("subiu", "O"), ("para", "O"), ("10,5%.", "B-MONEY"),
]

print("Texto:", texto_exemplo)
print("\nAnotação BIO:")
print(f"{'Token':<15} {'Tag':<10}")
print("-"*25)
for token, tag in tokens_bio:
    cor = ""
    if tag.startswith("B-"): cor = "  ← INÍCIO"
    elif tag.startswith("I-"): cor = "  ← CONTINUAÇÃO"
    print(f"{token:<15} {tag:<10}{cor}")


## 🤖 Passo 3 — NER com spaCy (modelo base)
Primeiro vamos ver o que o spaCy já consegue fazer com textos em português:


In [ ]:
import spacy

nlp = spacy.load("pt_core_news_lg")

textos_ner = [
    "O presidente do Banco Central, Roberto Campos Neto, reuniu-se com ministros em Brasília.",
    "A Petrobras anunciou lucro de R$ 36 bilhões no primeiro trimestre de 2026.",
    "O Itaú Unibanco, maior banco privado do Brasil, lançou o modelo NorBERTo para processamento de texto.",
    "Em São Paulo, a Fiesp alertou sobre os impactos da taxa de câmbio nas exportações.",
]

print("="*70)
print("NER com spaCy — pt_core_news_lg")
print("="*70)

for texto in textos_ner:
    doc = nlp(texto)
    print(f"\n📝 Texto: {texto}")
    if doc.ents:
        print("   Entidades encontradas:")
        for ent in doc.ents:
            print(f"   → [{ent.label_:8}] '{ent.text}'")
    else:
        print("   (nenhuma entidade detectada)")


## 🚀 Passo 4 — NER com NorBERTo (Token Classification)
Agora vamos usar um modelo BERT com classificação de tokens para NER em português. Usaremos um modelo NER pré-treinado que usa arquitetura BERT:


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# Modelo NER para português (baseado em BERT)
NER_MODEL = "lisaterumi/postagger-portuguese"

print(f"⏳ Carregando modelo NER: {NER_MODEL}")
ner_pipeline = pipeline("ner", model=NER_MODEL, aggregation_strategy="simple")
print("✅ Modelo NER carregado!")

def extrair_entidades_bert(texto):
    """Extrai entidades usando o pipeline NER do HuggingFace."""
    entidades = ner_pipeline(texto)
    return entidades

# Teste nos mesmos textos
print("\n" + "="*70)
print("NER com modelo BERT — Token Classification")
print("="*70)
for texto in textos_ner:
    ents = extrair_entidades_bert(texto)
    print(f"\n📝 Texto: {texto}")
    if ents:
        for ent in ents:
            score = ent.get('score', 0)
            print(f"   → [{ent['entity_group']:8}] '{ent['word']}' (conf: {score:.2f})")
    else:
        print("   (nenhuma entidade detectada)")


## 📊 Passo 5 — Extração de Informações do FAQ_BACEN
Vamos aplicar NER ao dataset real do Itaú-Unibanco para extrair entidades de textos regulatórios:


In [ ]:
from datasets import load_dataset
import re

print("⏳ Carregando FAQ_BACEN...")
faq = load_dataset("Itau-Unibanco/FAQ_BACEN", split="train")
print(f"✅ {len(faq)} registros carregados")
print(f"   Colunas: {faq.column_names}")

# Analisa entidades nas primeiras 20 perguntas
col_texto = faq.column_names[0]
textos_faq = [ex[col_texto] for ex in faq.select(range(20))]

print("\n" + "="*70)
print("Extração de Entidades — FAQ BACEN")
print("="*70)

entidades_acumuladas = {}
for texto in textos_faq:
    try:
        ents = ner_pipeline(texto[:200])  # limita tamanho
        for ent in ents:
            tipo = ent['entity_group']
            palavra = ent['word']
            if tipo not in entidades_acumuladas:
                entidades_acumuladas[tipo] = []
            entidades_acumuladas[tipo].append(palavra)
    except:
        pass

print("\nResumo das entidades encontradas no FAQ BACEN:")
for tipo, palavras in sorted(entidades_acumuladas.items()):
    unicas = list(set(palavras))[:5]
    print(f"  [{tipo}]: {unicas}")


## 🎨 Passo 6 — Visualização de Entidades com Highlight
Vamos criar uma visualização colorida das entidades identificadas:


In [ ]:
# Visualização HTML das entidades no Colab
from IPython.display import HTML

CORES_ENTIDADE = {
    "PER": "#FCD34D",   # Amarelo — pessoas
    "ORG": "#6EE7B7",   # Verde — organizações
    "LOC": "#93C5FD",   # Azul — locais
    "DATE": "#F9A8D4",  # Rosa — datas
    "MONEY": "#C4B5FD", # Roxo — valores
    "MISC": "#FCA5A5",  # Vermelho — outros
}

def destacar_entidades_html(texto, ents):
    """Cria HTML com entidades destacadas por cor."""
    # Ordena por posição de início
    ents_sorted = sorted(ents, key=lambda x: x['start'])
    html = ""
    cursor = 0
    for ent in ents_sorted:
        start, end = ent['start'], ent['end']
        tipo = ent.get('entity_group', ent.get('entity', 'UNK'))
        cor = CORES_ENTIDADE.get(tipo, "#E5E7EB")
        html += texto[cursor:start]
        html += (f'<mark style="background:{cor};padding:2px 4px;'
                 f'border-radius:3px;font-weight:bold" title="{tipo}">'
                 f'{texto[start:end]}<sup style="font-size:0.6em">{tipo}</sup></mark>')
        cursor = end
    html += texto[cursor:]
    return html

# Aplica nos textos de exemplo
for texto in textos_ner[:3]:
    try:
        ents = ner_pipeline(texto)
        html = destacar_entidades_html(texto, ents)
        display(HTML(f'<p style="font-family:Arial;font-size:14px;line-height:2">{html}</p>'))
        display(HTML('<hr>'))
    except Exception as e:
        print(f"Erro: {e}")

# Legenda
legenda = " ".join([
    f'<span style="background:{cor};padding:2px 6px;border-radius:3px">{tipo}</span>'
    for tipo, cor in CORES_ENTIDADE.items()
])
display(HTML(f'<p><b>Legenda:</b> {legenda}</p>'))


## 📝 Avaliação — Fase 4

**Q1.** O que significa NER em NLP?  
( ) Natural Expression Ranking  
( ) Named Entity Recognition  
( ) Neural Encoding Routine  
( ) Node Extraction Response

**Q2.** No formato BIO, o que significa a tag "B-ORG"?  
( ) O token é o final de uma organização  
( ) O token é o início de uma entidade do tipo organização  
( ) O token é uma organização de uma só palavra  
( ) O token está fora de qualquer entidade

**Q3.** Qual é a diferença entre classificação de sequência e classificação de tokens?  
( ) A classificação de sequência é mais precisa que a de tokens  
( ) Na sequência, uma label é atribuída à frase inteira; nos tokens, cada token recebe uma label  
( ) Na classificação de tokens não é possível usar BERT  
( ) A classificação de sequência usa o token [CLS], a de tokens usa [SEP]

**Q4.** O que é o parâmetro aggregation_strategy="simple" no pipeline NER?  
( ) Usa apenas o primeiro token de cada palavra  
( ) Agrupa tokens consecutivos da mesma entidade em uma só span  
( ) Filtra entidades com score abaixo de 0.5  
( ) Usa uma versão simplificada do modelo

**Q5.** Por que limitamos o texto a 200 caracteres na análise do FAQ_BACEN?  
( ) O modelo NER não suporta textos maiores  
( ) Para economizar memória e tempo no Colab  
( ) Porque textos maiores sempre têm mais erros de NER  
( ) O HuggingFace limita requisições longas

**Q6.** Qual das tags NER identifica nomes de pessoas?  
( ) LOC  ( ) ORG  ( ) PER  ( ) MISC

**Q7.** O token [CLS] em BERT para NER é usado para:  
( ) Identificar entidades do tipo data  
( ) Representação global da sequência (não recebe tag de entidade)  
( ) Marcar o início de cada entidade  
( ) Separar premissa e hipótese

**Q8.** Por que NER é importante para sistemas financeiros como os do Itaú-Unibanco?  
( ) Para calcular embeddings de documentos  
( ) Para extrair automaticamente entidades como valores, organizações e datas de documentos regulatórios  
( ) Para treinar modelos de geração de texto  
( ) Para classificar o sentimento de notícias financeiras

**Q9.** O modelo spaCy "pt_core_news_lg" foi treinado em qual tipo de texto?  
( ) Exclusivamente textos jurídicos  
( ) Notícias e textos web em português  
( ) Apenas tweets em português  
( ) Documentos do Banco Central do Brasil

**Q10.** No formato BIO, se temos "I-ORG" sem um "B-ORG" anterior, isso indica:  
( ) Uma entidade de organização de uma única palavra  
( ) Um erro de anotação — I deve sempre seguir B ou outro I da mesma entidade  
( ) O início de uma nova organização  
( ) Uma entidade aninhada (nested entity)
